In [1]:
!pip install transformers torch pandas tqdm -q

In [2]:
import torch
import pandas as pd
import numpy as np
from itertools import permutations
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import ast

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/sentence_prediction'

Mounted at /content/drive


In [4]:
train_df = pd.read_csv(f'{BASE_PATH}/train.csv')
print(f"train: {len(train_df)}개")
print(train_df.head(2))

train: 7351개
           ID                                         sentence_0  \
0  TRAIN_0000                 블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다.   
1  TRAIN_0001  줄거리 자동 생성의 인공지능 알고리즘은 대량의 텍스트 데이터를 분석하여 핵심 정보를...   

                                          sentence_1  \
0  이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다.   
1     결과적으로, 이러한 기술은 사용자에게 신속하고 효율적인 정보 전달을 가능하게 한다.   

                                          sentence_2  \
0  결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성...   
1     생성된 줄거리는 원본 텍스트의 의미를 유지하면서도 간결하게 요약된 형태로 제공된다.   

                                          sentence_3  answer_0  answer_1  \
0       각 투표는 변경 불가능한 기록으로 저장되어 조작의 가능성을 원천적으로 차단한다.         0         3   
1  이 알고리즘은 자연어 처리 기술을 활용하여 문맥을 이해하고, 주요 사건과 등장인물을...         0         3   

   answer_2  answer_3  
0         1         2  
1         2         1  


In [5]:
# PPL 원리 검증에 성능 가장 좋은 kanana 사용
MODEL_NAME = "kakaocorp/kanana-nano-2.1b-base"
SEPARATOR  = " "

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("로드 완료!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

로드 완료!


In [6]:
def calc_ppl(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    return torch.exp(outputs.loss).item()

In [7]:
print("=== PPL 원리 검증 ===\n")
print("정답 순서 vs 틀린 순서 PPL 비교\n")

# 샘플 100개로 검증
sample_df = train_df.sample(n=100, random_state=42)

correct_ppls = []  # 정답 순서 PPL
wrong_ppls   = []  # 틀린 순서 PPL 전체

for _, row in tqdm(sample_df.iterrows(), total=100):
    sentences = [row['sentence_0'], row['sentence_1'],
                 row['sentence_2'], row['sentence_3']]
    correct_order = [int(row['answer_0']), int(row['answer_1']),
                     int(row['answer_2']), int(row['answer_3'])]

    # 정답 순서 PPL
    correct_text = SEPARATOR.join([sentences[i] for i in correct_order])
    correct_ppl  = calc_ppl(correct_text)
    correct_ppls.append(correct_ppl)

    # 나머지 23가지 틀린 순서 PPL
    for perm in permutations(range(4)):
        if list(perm) != correct_order:
            wrong_text = SEPARATOR.join([sentences[i] for i in perm])
            wrong_ppl  = calc_ppl(wrong_text)
            wrong_ppls.append(wrong_ppl)

print(f"\n=== 결과 ===")
print(f"정답 순서 평균 PPL : {np.mean(correct_ppls):.2f}")
print(f"틀린 순서 평균 PPL : {np.mean(wrong_ppls):.2f}")
print(f"PPL 차이           : {np.mean(wrong_ppls) - np.mean(correct_ppls):.2f}")
print(f"\n정답이 더 낮은 비율: {np.mean(correct_ppls) < np.mean(wrong_ppls)}")

# 결과 저장
ppl_result = {
    'correct_ppl_mean': [np.mean(correct_ppls)],
    'wrong_ppl_mean'  : [np.mean(wrong_ppls)],
    'difference'      : [np.mean(wrong_ppls) - np.mean(correct_ppls)]
}
pd.DataFrame(ppl_result).to_csv(
    f'{BASE_PATH}/results/ppl_verification.csv',
    index=False
)
print("\n검증 결과 저장 완료!")

=== PPL 원리 검증 ===

정답 순서 vs 틀린 순서 PPL 비교



100%|██████████| 100/100 [01:37<00:00,  1.03it/s]


=== 결과 ===
정답 순서 평균 PPL : 4.02
틀린 순서 평균 PPL : 4.79
PPL 차이           : 0.77

정답이 더 낮은 비율: True

검증 결과 저장 완료!


In [9]:
print("=== 데이터 증강 시작 ===\n")

augmented_rows = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    sentences = [row['sentence_0'], row['sentence_1'],
                 row['sentence_2'], row['sentence_3']]
    correct_order = [int(row['answer_0']), int(row['answer_1']),
                     int(row['answer_2']), int(row['answer_3'])]

    # 24가지 순열 전부 생성
    for perm in permutations(range(4)):
        perm = list(perm)

        # 정답 여부 레이블
        is_correct = (perm == correct_order)

        augmented_rows.append({
            'original_id' : row['ID'],
            'sentence_0'  : sentences[0],
            'sentence_1'  : sentences[1],
            'sentence_2'  : sentences[2],
            'sentence_3'  : sentences[3],
            'order_0'     : perm[0],
            'order_1'     : perm[1],
            'order_2'     : perm[2],
            'order_3'     : perm[3],
            'is_correct'  : int(is_correct),  # 1=정답, 0=오답
            'ordered_text': SEPARATOR.join([sentences[i] for i in perm]),
            'answer_str'  : f"{perm[0]} {perm[1]} {perm[2]} {perm[3]}"
        })

augmented_df = pd.DataFrame(augmented_rows)

# 저장
save_path = f'{BASE_PATH}/augmented_train.csv'
augmented_df.to_csv(save_path, index=False)

print(f"원본 데이터   : {len(train_df)}개")
print(f"증강 후 데이터: {len(augmented_df)}개 (×24)")
print(f"저장 완료     : {save_path}")
print(augmented_df.head(3))

=== 데이터 증강 시작 ===



100%|██████████| 7351/7351 [00:01<00:00, 6466.16it/s]


원본 데이터   : 7351개
증강 후 데이터: 176424개 (×24)
저장 완료     : /content/drive/MyDrive/sentence_prediction/augmented_train.csv
  original_id                          sentence_0  \
0  TRAIN_0000  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다.   
1  TRAIN_0000  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다.   
2  TRAIN_0000  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다.   

                                          sentence_1  \
0  이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다.   
1  이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다.   
2  이러한 특성은 유권자들에게 신뢰를 제공하며, 민주적 참여를 촉진하는 데 기여할 수 있다.   

                                          sentence_2  \
0  결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성...   
1  결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성...   
2  결과적으로 블록체인 기반의 투표 시스템은 공정하고 신뢰할 수 있는 선거 환경을 조성...   

                                     sentence_3  order_0  order_1  order_2  \
0  각 투표는 변경 불가능한 기록으로 저장되어 조작의 가능성을 원천적으로 차단한다.        0        1        2   
1  각 투표는 변경 불가능한 기록으로 저장되어 조작의 가능성을 원천적으로 차단한다.        0        1

In [10]:
print("=== 증강 데이터 확인 ===\n")

# 정답/오답 비율
print(f"정답 샘플 수 : {augmented_df['is_correct'].sum()}")
print(f"오답 샘플 수 : {(augmented_df['is_correct'] == 0).sum()}")
print(f"전체 샘플 수 : {len(augmented_df)}")
print(f"\n샘플 확인:")
print(augmented_df[['ordered_text', 'is_correct', 'answer_str']].head(5))

=== 증강 데이터 확인 ===

정답 샘플 수 : 7351
오답 샘플 수 : 169073
전체 샘플 수 : 176424

샘플 확인:
                                        ordered_text  is_correct answer_str
0  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다. 이러한 특성은 유권자...           0    0 1 2 3
1  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다. 이러한 특성은 유권자...           0    0 1 3 2
2  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다. 결과적으로 블록체인 ...           0    0 2 1 3
3  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다. 결과적으로 블록체인 ...           0    0 2 3 1
4  블록체인 기술은 투표 과정의 투명성을 크게 향상시킬 수 있다. 각 투표는 변경 불가...           1    0 3 1 2
